# Individual ID Workflow

This notebook demonstrates the individual identification workflow on Eurasian otter data.


## 1. Load the dataset
We load the training split from `data/splits/eurasian_otter`. The helper `load_and_prep_df_individual` removes the `Fold` column, so we read the parquet file directly and rename the `Fold` field to `fold`.


In [ ]:

from pathlib import Path
import pandas as pd
from FIT_python.data_split_and_summary.data_import_utils import get_feature_cols

splits_root = Path('data/splits/eurasian_otter')
df = pd.read_parquet(splits_root / 'train.parquet').rename(columns={'Fold': 'fold'})
feature_cols = get_feature_cols(df)
print('rows:', len(df))
print('n_features:', len(feature_cols))


The sequential holdout sizes are defined by `SOFT_CONFIG['pipeline_individual_id']['sequential_holdout_val_sizes']` which defaults to [2, 4, 6, 8].

## 2. Generate pairwise comparisons
`generate_pairwise_comparisons_from_df` creates all trail pairs. The returned records include `same_fold`, `fold_a` and `fold_b`. Because our dataset has a `substrate` column the pairs also contain `substrate_a` and `substrate_b`.


In [ ]:

from FIT_python.pipeline_individual_id import generate_trails_and_trailpairs as gt

if not hasattr(gt, '_mode'):
    def _mode(series):
        return series.mode(dropna=True).iat[0] if not series.dropna().empty else pd.NA
    gt._mode = _mode

pairs, summary = gt.generate_pairwise_comparisons_from_df(df)
print(summary)
print(pairs[0])


## 3. Sequential holdout evaluation
We evaluate a single iteration using `sequential_holdout.run`. Population estimation is enabled on the train set and the predicted counts are compared to the true number of individuals in the validation set.


In [ ]:

from FIT_python.pipeline_individual_id.sequential_holdout import run
summary_df = run(df, feature_cols, iterations=1, val_sizes=[2], n_jobs=1)
summary_df


## 4. Inspect results
For each split we load the saved pairwise predictions and compute a confusion matrix as well as the Balanced Classification Rate (BCR). Additionally we plot a dendrogram of the pairwise distance matrix and show how the predicted population size compares with the true count using Lin's concordance correlation coefficient (CCC).


In [ ]:

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from FIT_python.pipeline_individual_id.evaluation import compute_confusion, compute_bcr
from FIT_python.pipeline_individual_id.population_estimation import concordance_correlation_coefficient
from scipy.cluster.hierarchy import linkage, dendrogram
from scipy.spatial.distance import squareform
from pathlib import Path

out_dir = Path('results/data/individual_id')
res = []
for split_file in sorted(out_dir.glob('split_*.csv')):
    df_res = pd.read_csv(split_file)
    cm = compute_confusion(df_res)
    bcr = compute_bcr(cm)
    display(cm)
    print('BCR:', bcr)
    trails = sorted(set(df_res['trail_a_id']) | set(df_res['trail_b_id']))
    dist = pd.DataFrame(float('inf'), index=trails, columns=trails)
    for a, b, val in zip(df_res['trail_a_id'], df_res['trail_b_id'], df_res['dist_euclidean']):
        dist.at[a, b] = val
        dist.at[b, a] = val
    np.fill_diagonal(dist.values, 0.0)
    Z = linkage(squareform(dist.values), method='ward')
    plt.figure(figsize=(6,2))
    dendrogram(Z, labels=trails, orientation='top')
    plt.show()
    true_n = df_res['ind_a'].nunique()
    pred_n = summary_df.loc[summary_df['split']==int(split_file.stem.split('_')[1]), 'pred_count'].iloc[0]
    res.append((pred_n, true_n))

if res:
    pred, true = zip(*res)
    plt.figure()
    plt.scatter(true, pred)
    ccc = concordance_correlation_coefficient(pred, true)
    plt.xlabel('true N')
    plt.ylabel('predicted N')
    plt.title(f'CCC = {ccc:.3f}')
    plt.show()
